预训练（CLM 错位 Loss）、监督微调（SFT 掩码机制）以及奖励模型（Pairwise 排序损失）是大模型的三大生命支柱。

近端策略优化PPO 极其强大，但它在工程上是一场灾难：它需要同时维护策略模型（Actor）、参考模型（Reference）、评论家模型（Critic）和奖励模型（Reward）四个庞大的模型，显存开销巨大，且强化学习的训练极不稳定、超参数极其敏感。
为了打破 PPO 的工程壁垒，直接偏好优化DPO成为工业界微调最优选择。

# 直接偏好优化（DPO），手写无 RM 的对齐损失函数
1. DPO 是如何通过数学魔法，省去显卡杀手 PPO 中的奖励模型与价值网络的？
2. 在代码上如何徒手实现 DPO 核心的隐式奖励差值与损失计算？

## DPO——从显性奖励到隐式奖励
PPO 算法中，大模型（Actor）生成回答，奖励模型（RM）打分，大模型根据分数调整参数。而 DPO 的核心思想非常震撼：大模型本身就可以隐式地充当自己的奖励模型！

通过严格的数学推导，DPO 证明了：奖励模型给一个回答 $y$ 打的分数 $r(x,y)$，可以完美地等价为当前模型与参考模型在生成该回答时的对数概率之差（Log-Ratio）。
$$\text{Implicit Reward } r(x, y) = \beta \log \frac{\pi_\theta(y\vert{}x)}{\pi_{\text{ref}}(y\vert{}x)}$$

* $\pi_\theta(y\vert{}x)$：我们正在训练的当前策略模型（Policy Model）生成该回答的概率。
* $\pi_{\text{ref}}(y\vert{}x)$：保持冻结的参考模型（Reference Model，通常是 SFT 后的底座）生成该回答的概率。
* $\beta$：一个控制 KL 散度惩罚的超参数（防止模型训歪，偏离初始底座太远）。

这意味着，我们不需要单独训练奖励模型了！我们只需要把 Chosen（好）和 Rejected（差）两个回答同时喂给“当前模型”和“参考模型”，算出各自的概率，就能直接利用偏好数据来优化当前模型。

## DPO 损失函数的工程形态
将隐式奖励代入 Pairwise 排序损失函数中，就得到了 DPO 的核心公式：
$$\mathcal{L}_{\text{DPO}}(\theta) = - \mathbb{E}_{(x, y_w, y_l) \sim D} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w\vert{}x)}{\pi_{\text{ref}}(y_w\vert{}x)} - \beta \log \frac{\pi_\theta(y_l\vert{}x)}{\pi_{\text{ref}}(y_l\vert{}x)} \right) \right]$$

让我们拆解它的物理意义：
1. 模型希望加大 Chosen（好回答）的隐式奖励，降低 Rejected（差回答）的隐式奖励。
2. 括号内的核心项是：$\text{Chosen 隐式奖励} - \text{Rejected 隐式奖励}$。
3. 如果当前模型让好回答的概率提升得比参考模型快，且让差回答的概率下降得快，差值就会变大，Loss 就会减小。

在工程上，我们只需要维护两个模型（Policy 和 Reference，其中 Reference 的梯度是冻结的），即可实现超越 PPO 的对齐效果！

我们用 PyTorch 模拟两条回答在 Policy 模型和 Reference 模型中的对数概率提取，并实现完整的 DPO Loss 计算。为了聚焦核心逻辑，我们直接模拟模型输出的 Logits 和对应的 SFT 掩码。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 辅助函数：计算一整条句子中 Response 部分的平均对数概率 (Log Probability)
def get_batch_logps(logits, labels, ignore_index=-100):
    """
    logits: (Batch, Seq_Len, Vocab_Size)
    labels: (Batch, Seq_Len) 带有 -100 掩码的真实标签
    """
    # 1. 执行错位对齐 (Shift)，自回归预测下一个词的标准操作
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    # 2. 计算每个位置对整个词表的 log_softmax
    # 形状: (Batch, Seq_Len-1, Vocab_Size)
    log_probs = F.log_softmax(shift_logits, dim=-1)

    # 3. 使用 gather 抽取出模型在“真实标签”位置上的 log 概率
    # 避开掩码: 把 -100 临时换成 0 以防 gather 越界报错
    loss_labels = shift_labels.clone()
    loss_labels[loss_labels == ignore_index] = 0

    # 抽出目标词的概率，形状: (Batch, Seq_Len-1)
    per_token_logps = log_probs.gather(dim=-1, index=loss_labels.unsqueeze(-1)).squeeze(-1)

    # 4. 利用掩码矩阵，将原本是 -100（Prompt部分）的位置的概率清零
    mask = (shift_labels != ignore_index)
    per_token_logps = per_token_logps * mask

    # 5. 对 Response 区域的 Token 概率求和，作为整句话的 Log 概率
    return per_token_logps.sum(dim=-1)

# --- 模拟 DPO 的损失计算步 ---
if __name__ == "__main__":
    torch.manual_seed(42)
    vocab_size = 10
    beta = 0.1  # DPO 超参数

    # 模拟输入：假设 Batch Size = 1（内含一条 chosen 和一条 rejected，拼在一起共 2 条数据）
    # 实际上为了算对数差， chosen 和 rejected 往往并排送入模型
    # 假设前 3 个词是 Prompt，后面是各自的 Response 且已对齐补齐
    # 真实 Label 中，Prompt 区域全部为 -100 掩码
    labels_chosen = torch.tensor([[ -100, -100, -100, 4, 5, 6 ]], dtype=torch.long)
    labels_reject = torch.tensor([[ -100, -100, -100, 7, 8, -100 ]], dtype=torch.long)

    # 模拟当前策略模型 (Policy) 吐出的 Logits
    policy_logits_chosen = torch.randn(1, 6, vocab_size)
    policy_logits_reject = torch.randn(1, 6, vocab_size)

    # 模拟冻结的底座模型 (Reference) 吐出的 Logits
    ref_logits_chosen = torch.randn(1, 6, vocab_size)
    ref_logits_reject = torch.randn(1, 6, vocab_size)

    # 1. 分别提取 Policy 模型对 Chosen 和 Rejected 的全句对数概率
    policy_chosen_logps = get_batch_logps(policy_logits_chosen, labels_chosen)
    policy_reject_logps = get_batch_logps(policy_logits_reject, labels_reject)

    # 2. 分别提取 Reference 模型对 Chosen 和 Rejected 的全句对数概率
    ref_chosen_logps = get_batch_logps(ref_logits_chosen, labels_chosen)
    ref_reject_logps = get_batch_logps(ref_logits_reject, labels_reject)

    print(f"Policy   -> Chosen LogP: {policy_chosen_logps.item():.4f}, Reject LogP: {policy_reject_logps.item():.4f}")
    print(f"Reference-> Chosen LogP: {ref_chosen_logps.item():.4f}, Reject LogP: {ref_reject_logps.item():.4f}")

    # 3. 计算隐式奖励 (Implicit Rewards)
    chosen_rewards = beta * (policy_chosen_logps - ref_chosen_logps)
    reject_rewards = beta * (policy_reject_logps - ref_reject_logps)

    print(f"\n【隐式奖励计算结果】Chosen 奖励: {chosen_rewards.item():.4f}, Reject 奖励: {reject_rewards.item():.4f}")

    # 4. 核心工程实现：DPO Loss 计算
    # 期望 chosen_rewards - reject_rewards 越大越好
    logits_diff = chosen_rewards - reject_rewards
    loss = -F.logsigmoid(logits_diff).mean()

    print(f"\n【计算出的最终 DPO 损失值】: {loss.item():.4f}")

1. 走通 gather 提取对数概率的工程逻辑：在 `get_batch_logps` 函数中，我们使用了 `log_probs.gather` 这一非常硬核的张量操作。请结合第 2 天的 SFT 知识解释：为什么我们在提取对数概率时，必须再次引入 `mask = (shift_labels != ignore_index)` 的乘法操作？如果不进行这一步掩码清零，全句求和出来的 logps 会夹杂进哪些不属于大模型回答区域的“脏随机数”？对最终 DPO 的优化会造成什么后果？
2. 理解 DPO 对概率分布的隐式拉扯：观察 DPO Loss 的形态。假设在训练初期，Policy 模型表现很差，把 Chosen（好回答）的概率降得极低（相比 Reference 变成负数），而把 Rejected（差回答）的概率升得很高。此时 logits_diff 会变成一个很大的负数。
    * 这时 `-F.logsigmoid(logits_diff)` 会输出一个极大的 Loss 还是极小的 Loss？
    * 这个梯度在反向传播时，会迫使当前 Policy 模型对 Chosen 回答的 Token 概率做“拉高”还是“压低”操作？对 Rejected 回答的 Token 概率做什么操作？这如何体现了 DPO 独特的对比学习（Contrastive Learning）本质？

我们从底层的预训练（Pre-train）一路攻克到监督微调（SFT）和强化学习人类偏好对齐（RM 与 DPO）。至此，你已经完整走过了全参数/主流对齐算法的底层数据流和损失函数计算。

然而，在工业界实际落地时，如果我们要微调一个 7B（70亿参数）甚至 70B 的大模型，动辄需要数十 GB 甚至数百 GB 的显存。如果对全量参数进行更新，普通的消费级显卡（如 RTX 3090/4090）会直接爆显存（OOM）。为了打破算力垄断，参数高效微调（PEFT, Parameter-Efficient Fine-Tuning）应运而生。

深度拆解大模型微调的“降维打击”——LoRA（Low-Rank Adaptation，低秩适应）的数学原理与全套 PyTorch 底层徒手实现！

## LoRA低秩矩阵微调
1. LoRA 是如何利用“低秩矩阵分解”将数十亿的参数训练量降低 99% 以上的？
2. 在 PyTorch 中，如何不依赖 Hugging Face 的 peft 库，手动写出一个支持旁路全连接（Linear）的 LoRA 层？

#### LoRA 的核心思想——低秩分解与旁路矩阵
在大模型微调过程中，模型的知识更新其实可以表达为权重矩阵的改变量 $\Delta W$。对于一个预训练好的高维矩阵 $W_0 \in \mathbb{R}^{d \times k}$，全参数微调会直接修改这整个矩阵。

LoRA 提出了一项关键假设：大模型在特定任务上的权重更新（$\Delta W$）实际上具有很低的“内在秩（Intrinsic Rank）”。这意味着，我们不需要直接训练一个庞大的 $\Delta W$，而是可以把它拆解为两个低秩矩阵 $A$ 和 $B$ 的乘积：
$$\Delta W = B \times A$$

其中：
* $W_0 \in \mathbb{R}^{d \times k}$ （原始矩阵，完全冻结，不参与训练）
* $B \in \mathbb{R}^{d \times r}$ （旁路下采样矩阵，可训练）
* $A \in \mathbb{R}^{r \times k}$ （旁路拉升采样矩阵，可训练）
* $r$ 是设置的秩（Rank），通常非常小（如 $r=4$ 或 $r=8$），而 $d$ 和 $k$ 通常是几千（如 4096）。

**为什么这样能暴省显存？**
假设 $d=4096, k=4096$：
* 全参数微调：需要为 $\Delta W$ 维护 $4096 \times 4096 \approx 16,777,216$（约 1600 万）个参数的梯度和优化器状态。
* LoRA 微调（假设秩 $r=8$）：
    * 矩阵 $A$ 的参数量：$8 \times 4096 = 32,768$
    * 矩阵 $B$ 的参数量：$4096 \times 8 = 32,768$
    * 总可训练参数量：$32,768 + 32,768 = 65,536$（约 6.5 万）。
    * 参数量及显存占用直接暴跌了 99.6%！

#### LoRA 的初始化与缩放因子（Scaling Factor）

在前向传播时，LoRA 层的数学公式为：
$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} (BA)x$$

这里有两个极其关键的工程细节：
1. 矩阵初始化（零初始化的妙用）：
    * 矩阵 $A$ 通常采用高斯分布随机初始化。
    * 矩阵 $B$ 必须全部初始化为 0。
    * 为什么？ 因为在训练的第一步，$\Delta W = B \times A = 0 \times A = 0$。这样可以保证在微调刚开始的瞬间，LoRA 旁路没有任何副作用，模型的输出完全等价于原始预训练模型，从而保证了训练的稳定性。
2. 缩放因子 $\frac{\alpha}{r}$：
    * $\alpha$ 是一个常数超参数（Scaling Hyperparameter）。
    * 当我们尝试调整不同的秩 $r$ 时，缩放因子能够保持显式学习率的相对稳定，不需要每次改 $r$ 都重新大改学习率。

我们用 PyTorch 纯手工编写一个包含 LoRA 旁路的自定义线性层。这个实验将展示如何冻结原参数、初始化旁路、并在前向传播中融合低秩矩阵。

In [ ]:
import torch
import torch.nn as nn
import math

class LoraLinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16):
        super().__init__()
        # 1. 创建标准的预训练线性层（模拟底座模型的某一层）
        self.linear = nn.Linear(in_features, out_features)

        # 2. 核心工程动作：完全冻结原始模型的参数！不让它们计算梯度
        for param in self.linear.parameters():
            param.requires_grad = False

        self.r = r
        self.lora_alpha = lora_alpha
        # 缩放因子 alpha / r
        self.scaling = lora_alpha / r

        # 3. 构建低秩可训练旁路矩阵 A 和 B
        if r > 0:
            # 矩阵 A: 从 输入维度 降维到 秩 r
            self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
            # 矩阵 B: 从 秩 r 升维到 输出维度
            self.lora_B = nn.Parameter(torch.zeros((out_features, r)))

            # 4. 矩阵初始化
            # A 矩阵用凯明/高斯分布初始化
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            # B 矩阵必须严格归零！确保训练步 0 时旁路输出为 0
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        # x 形状: (batch_size, seq_len, in_features)

        # 基础路：走原有的冻结权重矩阵
        base_output = self.linear(x)

        # 旁路：走 LoRA 低秩拉扯路径
        if self.r > 0:
            # 根据矩阵乘法结合律，先让 x 与 lora_A 转置相乘，再与 lora_B 转置相乘
            # 数学等价于 x @ (B @ A).T，但分开相乘能大幅降低中间计算复杂度
            lora_output = (x @ self.lora_A.t()) @ self.lora_B.t()

            # 结合缩放因子，将旁路结果叠加回基础输出中
            return base_output + lora_output * self.scaling

        return base_output

# --- 验证 LoRA 层的初始化与梯度行为 ---
if __name__ == "__main__":
    torch.manual_seed(42)

    # 模拟输入维度：4096 (大模型常见维度)，输出维度: 4096
    in_dim, out_dim = 4096, 4096

    # 实例化我们的徒手 LoRA 层
    lora_layer = LoraLinear(in_features=in_dim, out_features=out_dim, r=8, lora_alpha=16)

    # 验证参数冻结情况
    total_params = sum(p.numel() for p in lora_layer.parameters())
    trainable_params = sum(p.numel() for p in lora_layer.parameters() if p.requires_grad)

    print(f"【LoRA 架构验证】")
    print(f"总参数量 (包含冻结的底座): {total_params:,}")
    print(f"实际可训练参数量 (仅LoRA旁路): {trainable_params:,}")
    print(f"参数量压缩比例: 仅需训练原参数的 {trainable_params / total_params * 100:.3f}% \n")

    # 模拟一个 Batch 的输入数据 (Batch=2, Seq_len=4, Dim=4096)
    simulated_input = torch.randn(2, 4, in_dim)

    # 前向传播
    output = lora_layer(simulated_input)

    # 验证步 0 的无缝对齐：此时由于 lora_B 全为 0，lora_layer 的输出必须和原生 linear 完全一样
    native_output = lora_layer.linear(simulated_input)
    is_identical = torch.allclose(output, native_output, atol=1e-6)
    print(f"【步 0 等价性测试】LoRA 输出是否与原冻结模型完全一致: {is_identical}")

    # 模拟一次反向传播
    loss = output.sum()
    loss.backward()

    # 检查梯度：底座参数的梯度必须是 None，只有 lora_A 和 lora_B 有梯度
    base_weight_grad = lora_layer.linear.weight.grad
    lora_A_grad = lora_layer.lora_A.grad

    print(f"\n【梯度传导检查】")
    print(f"底座权重矩阵的梯度是否成功被冻结 (应为 None): {base_weight_grad is None}")
    print(f"LoRA 旁路矩阵 A 的梯度是否成功生成: {lora_A_grad is not None and lora_A_grad.sum() != 0}")

1. 深度理解矩阵乘法的计算复杂度降低：在 `forward` 代码中，我们写的是 `(x @ self.lora_A.t()) @ self.lora_B.t()`。请你推导一下以下两者的计算复杂度差异：
    * 方式一：先将低秩矩阵相乘恢复为满秩改变量，即先算 $W_{new} = B \times A$（维度为 $d \times k$），再用 $x \times W_{new}^T$。
    * 方式二：代码中的串联方式，先算 $x \times A^T$，再用结果乘以 $B^T$。
    * 结合秩 $r=8$ 极其微小这一事实，想一想为什么方式二在运行效率和显存节约上具有绝对的压倒性优势？

2. 权重合并（Weight Merge）的上线思考：虽然 LoRA 在训练时通过旁路完美省下了显存，但在工业级线上高并发推理（Inference）时，如果每次前向传播都要让数据走两遍矩阵（原生矩阵一遍，旁路一遍再相加），这会带来无法接受的额外推理延迟（Latency）。
    * 提示：根据矩阵乘法的分配律，$W_0 x + \Delta W x = (W_0 + \Delta W) x$。
    * 请问在模型训练结束后，我们应该在工程上对 lora_layer.linear.weight 做什么操作，从而将模型恢复成一个没有任何旁路结构的、普通的、标准的深度学习模型？这样做为什么能实现“训练时省显存，推理时不增加任何计算延迟”的完美闭环？